# End-to-End AI Research-Paper Summarization

This notebook implements the assignment as a runnable prototype using an actual research paper:

- PDF text extraction
- Cleaning and section detection
- Long-document chunking
- Abstractive summarization with a T5/BART/PEGASUS model
- Structured extraction of objective, methodology, dataset, findings, and conclusion
- ROUGE-1, ROUGE-2, ROUGE-L, BLEU, and perplexity

The default input is the ArXiv paper *Attention Is All You Need*. Change `PDF_PATH` to summarize another ArXiv or PubMed paper.

In [1]:
# Install missing packages into a local notebook directory, not system Python.
import importlib.util
import subprocess
import sys
from pathlib import Path

LOCAL_PACKAGE_DIR = Path('.notebook_packages')
LOCAL_PACKAGE_DIR.mkdir(exist_ok=True)
if str(LOCAL_PACKAGE_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(LOCAL_PACKAGE_DIR.resolve()))

if importlib.util.find_spec('pypdf') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--target', str(LOCAL_PACKAGE_DIR), 'pypdf'])

missing_transformer_packages = [
    package for module, package in {'transformers': 'transformers', 'sentencepiece': 'sentencepiece', 'torch': 'torch'}.items()
    if importlib.util.find_spec(module) is None
]
if missing_transformer_packages and sys.version_info >= (3, 14):
    raise RuntimeError(
        'This notebook requires Python 3.11-3.13 for the Transformer model. '
        f'Your kernel is Python {sys.version_info.major}.{sys.version_info.minor}. '
        'Create/select a Python 3.13 Jupyter kernel, then rerun the notebook.'
    )
if missing_transformer_packages:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--target', str(LOCAL_PACKAGE_DIR), *missing_transformer_packages])
    except subprocess.CalledProcessError as exc:
        raise RuntimeError('Transformer dependencies could not be installed. Use a Python 3.11-3.13 kernel.') from exc

import math
import re
import ssl
import urllib.request
from collections import Counter
from typing import Dict, List

from pypdf import PdfReader

PDF_PATH = Path('attention_is_all_you_need.pdf')
PAPER_URL = 'https://arxiv.org/pdf/1706.03762'
MODEL_NAME = 'google/flan-t5-base'  # T5 encoder-decoder model required by the assignment
MAX_CHUNK_WORDS = 650

def download_research_paper(url: str, destination: Path):
    try:
        import certifi
        context = ssl.create_default_context(cafile=certifi.where())
    except Exception:
        context = ssl.create_default_context()
    try:
        with urllib.request.urlopen(url, context=context) as response:
            destination.write_bytes(response.read())
    except Exception as verified_error:
        # Some corporate networks replace public certificates with an internal CA.
        print('Verified download failed:', verified_error)
        print('Retrying the fixed public ArXiv URL with certificate verification disabled.')
        insecure_context = ssl._create_unverified_context()
        with urllib.request.urlopen(url, context=insecure_context) as response:
            destination.write_bytes(response.read())

if not PDF_PATH.exists():
    print('Downloading research paper:', PAPER_URL)
    download_research_paper(PAPER_URL, PDF_PATH)
assert PDF_PATH.exists(), f'Research paper not found: {PDF_PATH}'
print('Input:', PDF_PATH)

Verified download failed: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1032)>
Retrying the fixed public ArXiv URL with certificate verification disabled.
Input: attention_is_all_you_need.pdf


## 1. PDF extraction and preprocessing

The extractor preserves page boundaries and removes common PDF artifacts. Scanned PDFs may require OCR before this notebook can process them.

In [2]:
def extract_pdf_pages(pdf_path: Path) -> List[str]:
    reader = PdfReader(str(pdf_path))
    pages = []
    for page in reader.pages:
        text = page.extract_text() or ''
        pages.append(text)
    return pages

def clean_text(text: str) -> str:
    text = text.replace('\u00ad', '')
    text = re.sub(r'(?<=\w)-\s*\n\s*(?=\w)', '', text)
    text = re.sub(r'\s*\n\s*', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def extract_pdf_text(pdf_path: Path):
    pages = extract_pdf_pages(pdf_path)
    cleaned_pages = [clean_text(p) for p in pages]
    return '\n'.join(cleaned_pages), cleaned_pages

raw_text, page_text = extract_pdf_text(PDF_PATH)
print(f'Pages: {len(page_text)} | Characters: {len(raw_text):,} | Words: {len(raw_text.split()):,}')
print(raw_text[:900])

Pages: 15 | Characters: 39,576 | Words: 6,107
Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Parmar∗ Google Research nikip@google.com Jakob Uszkoreit∗ Google Research usz@google.com Llion Jones∗ Google Research llion@google.com Aidan N. Gomez∗ † University of Toronto aidan@cs.toronto.edu Łukasz Kaiser∗ Google Brain lukaszkaiser@google.com Illia Polosukhin∗ ‡ illia.polosukhin@gmail.com Abstract The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transfor


## 2. Section detection and long-document chunking

Standard encoder-decoder models cannot reliably consume an entire long paper at once. The implementation therefore detects common scientific sections, splits text into bounded chunks, summarizes chunks, and combines the results.

In [3]:
SECTION_NAMES = [
    'abstract', 'introduction', 'background', 'related work', 'methodology',
    'methods', 'materials and methods', 'experiments', 'results',
    'discussion', 'conclusion', 'limitations', 'references'
]

def split_sentences(text: str) -> List[str]:
    text = re.sub(r'\s+', ' ', text).strip()
    if not text:
        return []
    parts = re.split(r'(?<=[.!?])\s+(?=[A-Z0-9])', text)
    return [p.strip() for p in parts if len(p.strip()) > 20]

def detect_sections(text: str) -> Dict[str, str]:
    normalized = text
    matches = []
    for name in SECTION_NAMES:
        pattern = r'(?i)(?<![A-Za-z])' + re.escape(name) + r'(?![A-Za-z])'
        for m in re.finditer(pattern, normalized):
            prefix = normalized[max(0, m.start()-35):m.start()]
            if m.start() == 0 or prefix.endswith(('.', ':', ' ')):
                matches.append((m.start(), name))
    matches.sort()
    sections = {}
    for i, (start, name) in enumerate(matches):
        end = matches[i+1][0] if i + 1 < len(matches) else len(normalized)
        content = normalized[start + len(name):end].strip(' :.-')
        if len(content.split()) >= 8 and name not in sections:
            sections[name] = content
    if not sections:
        sections['document'] = normalized
    return sections

def chunk_text(text: str, max_words: int = 650) -> List[str]:
    sentences = split_sentences(text)
    chunks, current, count = [], [], 0
    for sentence in sentences:
        words = len(sentence.split())
        if current and count + words > max_words:
            chunks.append(' '.join(current))
            current, count = [], 0
        current.append(sentence)
        count += words
    if current:
        chunks.append(' '.join(current))
    return chunks

sections = detect_sections(raw_text)
chunks = []
for section, content in sections.items():
    for chunk in chunk_text(content, MAX_CHUNK_WORDS):
        chunks.append({'section': section, 'text': chunk})
print('Detected sections:', list(sections))
print('Chunks:', len(chunks))

Detected sections: ['abstract', 'experiments', 'results', 'introduction', 'background', 'conclusion', 'references', 'methods']
Chunks: 10


## 3. Summarization engine

The notebook uses a pretrained Transformer encoder-decoder model. Set `MODEL_NAME` to a compatible T5, BART, or PEGASUS checkpoint.

In [4]:
class TransformerSummarizer:
    def __init__(self, model_name: str):
        from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
        import torch
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model.to(self.device)
        self.model.eval()

    def summarize(self, text: str, max_new_tokens: int = 180) -> str:
        model_name = self.model.config.name_or_path.lower()
        prompt = text if ('bart' in model_name or 'pegasus' in model_name) else 'summarize: ' + text
        inputs = self.tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024).to(self.device)
        output = self.model.generate(**inputs, max_new_tokens=max_new_tokens, num_beams=4, no_repeat_ngram_size=3)
        return self.tokenizer.decode(output[0], skip_special_tokens=True).strip()

model = TransformerSummarizer(MODEL_NAME)
print('Loaded Transformer:', MODEL_NAME)

def summarize_chunks(chunk_records):
    summaries = []
    for record in chunk_records:
        summary = model.summarize(record['text'])
        summaries.append({'section': record['section'], 'summary': summary})
    combined = ' '.join(x['summary'] for x in summaries)
    final = model.summarize(combined)
    return final, summaries

document_summary, chunk_summaries = summarize_chunks(chunks)
print(document_summary)

/Users/yogeshtolani/Library/Python/3.13/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1032)' thrown while requesting HEAD https://huggingface.co/google/flan-t5-base/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1032)' thrown while requesting HEAD https://huggingface.co/google/flan-t5-base/resolve/main/config.json
Retrying in 2s [Retry 2/5].
'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1032)' thrown while requesting HEAD https://huggingface.co/google/flan-t5-base/resolve/main/config.json
Retrying in 4s [Retr

OSError: We couldn't connect to 'https://huggingface.co' to load the files, and couldn't find them in the cached files.
Check your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#offline-mode'.

## 4. Structured research-information extraction

The extractor returns the research objective, methodology, dataset used, key findings, and conclusion required by the assignment.

In [ ]:
FIELD_RULES = {
    'research_objective': ['objective', 'aim', 'we propose', 'we investigate', 'we study', 'this paper'],
    'methodology': ['method', 'approach', 'architecture', 'algorithm', 'experiment', 'model'],
    'dataset_used': ['dataset', 'data set', 'corpus', 'benchmark', 'participants', 'samples'],
    'key_findings': ['result', 'findings', 'achieve', 'outperform', 'improve', 'accuracy', 'significant'],
    'conclusion': ['conclusion', 'conclude', 'overall', 'future work', 'limitation']
}

def extract_structured_fields(text: str) -> Dict[str, Dict[str, str]]:
    all_sentences = split_sentences(text)
    result = {}
    for field, keywords in FIELD_RULES.items():
        candidates = []
        for i, sentence in enumerate(all_sentences):
            lower = sentence.lower()
            hits = sum(1 for keyword in keywords if keyword in lower)
            if hits:
                candidates.append((hits, -i, sentence))
        candidates.sort(reverse=True)
        evidence = [x[2] for x in candidates[:3]]
        if not evidence:
            evidence = ['Information not confidently detected in the extracted text.']
        result[field] = {'value': ' '.join(evidence), 'evidence': evidence}
    return result

structured_fields = extract_structured_fields(raw_text)
for field, item in structured_fields.items():
    print('\n' + field + ':\n' + item['value'])

## 5. Evaluation metrics

The evaluation implements the five metrics listed in the assignment. Set `REFERENCE` to the author-provided summary or abstract.

In [ ]:
def words(text):
    return re.findall(r'\w+', text.lower())

def rouge_n(reference: str, candidate: str, n: int = 1) -> float:
    def grams(tokens):
        return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1))
    ref, cand = grams(words(reference)), grams(words(candidate))
    overlap = sum((ref & cand).values())
    precision = overlap / max(1, sum(cand.values()))
    recall = overlap / max(1, sum(ref.values()))
    return 2 * precision * recall / max(1e-12, precision + recall)

def rouge_l(reference: str, candidate: str) -> float:
    a, b = words(reference), words(candidate)
    table = [[0] * (len(b)+1) for _ in range(len(a)+1)]
    for i in range(1, len(a)+1):
        for j in range(1, len(b)+1):
            table[i][j] = table[i-1][j-1] + 1 if a[i-1] == b[j-1] else max(table[i-1][j], table[i][j-1])
    lcs = table[-1][-1]
    precision, recall = lcs / max(1, len(b)), lcs / max(1, len(a))
    return 2 * precision * recall / max(1e-12, precision + recall)

def bleu(reference: str, candidate: str) -> float:
    ref, cand = words(reference), words(candidate)
    if not cand: return 0.0
    overlap = sum((Counter(ref) & Counter(cand)).values())
    precision = overlap / len(cand)
    brevity = min(1.0, math.exp(1 - len(ref) / max(1, len(cand))))
    return precision * brevity

def perplexity(source: str, reference: str) -> float:
    import torch
    source_inputs = model.tokenizer(source, return_tensors='pt', truncation=True, max_length=1024).to(model.device)
    target_inputs = model.tokenizer(reference, return_tensors='pt', truncation=True, max_length=512).to(model.device)
    with torch.no_grad():
        loss = model.model(**source_inputs, labels=target_inputs['input_ids']).loss
    return float(torch.exp(loss).cpu())

def evaluate_summary(reference: str, candidate: str) -> Dict[str, float]:
    return {
        'rouge_1_f1': rouge_n(reference, candidate, 1),
        'rouge_2_f1': rouge_n(reference, candidate, 2),
        'rouge_l_f1': rouge_l(reference, candidate),
        'bleu_score': bleu(reference, candidate),
        'perplexity': perplexity(raw_text, reference)
    }

# The paper abstract is the author-provided reference summary.
REFERENCE = sections.get('abstract', '')
if not REFERENCE:
    raise ValueError('No abstract/reference summary was detected in the research paper.')
print(evaluate_summary(REFERENCE, document_summary))